# Test Epic_clinical_notes

In [ ]:
import os
import random
import shutil
import sys

import numpy as np

In [ ]:
_here = os.getcwd()
for _c in (_here, os.path.join(_here, "notebooks", "test")):
    if os.path.exists(os.path.join(_c, "nb_temp_setup.py")):
        if _c not in sys.path:
            sys.path.insert(0, _c)
        break
else:
    raise FileNotFoundError(
        "nb_temp_setup.py not found in the working directory or "
        "notebooks/test. Run this notebook from notebooks/test or the "
        "repository root."
    )

from nb_temp_setup import cleanup_nb_temp_dir, setup_nb_temp_dir

nb_temp_dir, repo_root = setup_nb_temp_dir()

random_seed_value = 42

np.random.seed(random_seed_value)
random.seed(random_seed_value)

print(f"Temp directory: {nb_temp_dir}")

In [ ]:
from pat2vec.util.config_pat2vec import config_class

schema_path = os.path.abspath("../test_files/elastic_schemas.json")

config_populate = config_class(
    proj_name="epic_clinical_notes_test_project",
    credentials_path=creds_filename,
    test_schema_path=schema_path,
    testing=True,
    testing_elastic=True,
    global_start_year=2020,
    global_start_month=1,
    global_start_day=1,
    global_end_year=2023,
    global_end_month=12,
    global_end_day=31,
)

In [ ]:
from pat2vec.util.get_dummy_data_cohort_searcher import populate_elastic_with_dummy_data

print("Populating test Elasticsearch cluster with dummy data...")
patient_ids = populate_elastic_with_dummy_data(config_populate, n_patients=5)
print(f"Population complete. Generated {len(patient_ids)} dummy patients.")

In [ ]:
from pat2vec.pat2vec_search.cogstack_search_methods import initialize_cogstack_client

cs = initialize_cogstack_client(config_populate)

indices = ["epr_documents", "basic_observations", "observations", "order", "pims_apps"]
print("Refreshing indices...")
cs.elastic.indices.refresh(index=indices, ignore_unavailable=True)
import time

time.sleep(2)
print("Indices refreshed.")

In [ ]:
PROJ_NAME = "epic_clinical_notes_test_project"
DB_FILENAME = "temp_epic_clinical_notes_db.sqlite"
DB_PATH = os.path.join(PROJ_NAME, "outputs", DB_FILENAME)

os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)

In [ ]:
from pat2vec.util.logger_setup import setup_logger

logger = setup_logger()
print("Logger initialized.")

In [ ]:
from pat2vec.main_pat2vec import main

config_obj = config_class(
    proj_name=PROJ_NAME,
    credentials_path=creds_filename,
    current_path_dir="",
    main_options={"epic_clinical_notes_annotations": True},
    all_patient_list=patient_ids,
    batch_mode=True,
    verbosity=0,
    random_seed_val=random_seed_value,
    testing=True,
    testing_elastic=True,
    dummy_medcat_model=True,
    use_controls=False,
    medcat=False,
    start_time=None,
    patient_id_column_name="client_idcode",
    annot_filter_options={},
    shuffle_pat_list=False,
    storage_backend="database",
    db_connection_string=db_connection_string,
    check_patient_existence=False,
    treatment_doc_filename="test_files/treatment_docs.csv",
)
pat2vec_obj = main(
    cogstack=True,
    use_filter=False,
    json_filter_path=None,
    random_seed_val=random_seed_value,
    hostname=None,
    config_obj=config_obj,
)

In [ ]:
print("\n=== PROCESSING PATIENTS WITH pat_maker ===")
print(f"Patient list: {pat2vec_obj.all_patient_list}")

try:
    print(f"Processing patient 0: {pat2vec_obj.all_patient_list[0]}")
    pat2vec_obj.pat_maker(0)
except Exception as e:
    msg = f"Failed to process patient 0 with pat_maker: {e}. Critical error - pipeline failed."
    raise RuntimeError(
        msg,
    ) from e
print("Patient feature extraction complete.")

In [ ]:
from pat2vec.util.helper_functions import get_all_features

all_features = get_all_features(config_obj)

if all_features.empty:
    msg = "FATAL ERROR: get_all_features returned empty DataFrame"
    raise RuntimeError(msg)
print(f"Successfully retrieved {len(all_features)} rows from database.")

In [ ]:
import pandas as pd

from pat2vec.util.post_processing_build_methods import merge_epic_clinical_notes_csv

print("\n=== DEMONSTRATING DOCUMENTS MERGE FUNCTIONALITY ===")

merged_docs_path = merge_epic_clinical_notes_csv(
    pat2vec_obj.all_patient_list,
    config_obj,
    overwrite=True,
)

assert os.path.exists(merged_docs_path), (
    f"FATAL ERROR: Merged documents file should exist at {merged_docs_path}. "
    "Critical error - merge builder failed to create the documents file."
)

merged_docs = pd.read_csv(merged_docs_path)

if merged_docs.empty:
    raise RuntimeError(
        "FATAL ERROR: Merged documents dataframe is empty. "
        "This indicates the pat2vec pipeline did not save epic clinical notes "
        "documents to the database.",
    )

print(f"Merged documents data saved to: {merged_docs_path}")
print(f"Shape: {merged_docs.shape}")
print(f"\nColumns: {list(merged_docs.columns)}")
print("\nData preview:")
print(merged_docs.head())

In [ ]:
# === VECTOR VALIDATION ===
feature_cols = [c for c in all_features.columns if c.startswith("text_")]

assert len(feature_cols) > 0, "No feature columns found. Available columns: " + str(
    list(all_features.columns)
)

non_null_counts = all_features[feature_cols].notna().sum()
totally_empty = non_null_counts[non_null_counts == 0]

assert len(totally_empty) == 0, (
    f"The following feature columns are entirely null:\n"
    f"{list(totally_empty.index)}\n"
    "Vectorisation is silently failing — check the get method return value."
)

print("Feature columns (" + str(len(feature_cols)) + "): " + str(feature_cols))
print("Non-null counts per feature column:")
for col in sorted(feature_cols):
    val = all_features[col].notna().sum()
    print("  " + str(col) + ": " + str(val) + " non-null values")

In [ ]:
import pandas as pd

from pat2vec.util.post_processing_build_methods import build_merged_epr_mct_annot_df

print("\n=== DEMONSTRATING ANNOTATIONS MERGE FUNCTIONALITY ===")

merged_annots_path = build_merged_epr_mct_annot_df(
    pat2vec_obj.all_patient_list,
    config_obj,
    overwrite=True,
)

assert merged_annots_path is not None and os.path.exists(merged_annots_path), (
    f"FATAL ERROR: Merged annotations file should exist at {merged_annots_path}. "
    "Critical error - merge builder failed to create the annotations file."
)

merged_annots = pd.read_csv(merged_annots_path)

if merged_annots.empty:
    raise RuntimeError(
        "FATAL ERROR: Merged annotations dataframe is empty. "
        "This indicates the pat2vec pipeline did not save epic clinical notes "
        "annotations to the database.",
    )

print(f"Merged annotations data saved to: {merged_annots_path}")
print(f"Shape: {merged_annots.shape}")
print(f"\nColumns: {list(merged_annots.columns)}")
print("\nData preview:")
print(merged_annots.head())

In [ ]:
# Check pat_maker output above this cell for any of these strings.
# If any appear, the merge builders below will return empty data.
table_error_strings = [
    "no such table",
    "operationalerror",
    "table not found",
    "no table named",
]
print("=== TABLE ERROR CHECK ===")
print(
    "If pat_maker output above contains any of these strings, "
    "the merge builders will fail:",
)
for s in table_error_strings:
    print(f"  - '{s}'")
print("Proceeding to merge builders...")

In [ ]:
cleanup_nb_temp_dir(nb_temp_dir)

print("\n=== FINAL VERIFICATION ===")
assert not os.path.exists(DB_PATH), "Database file still exists!"
assert not os.path.exists(PROJ_NAME), "Project directory still exists!"
assert not os.path.exists(creds_filename), "Credentials file still exists!"

print("All cleanup verified - no residual files remain.")
print("\n=== TEST SUCCESSFUL ===")